# Setup

In [1]:
import pandas as pd
import numpy as np
import requests
import os
import re
import pickle

from glob import glob
from collections import Counter

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data

## Downloading The Data

In [3]:
parquet_urls = requests.get('https://huggingface.co/api/datasets/google/civil_comments/parquet/default').json()
parquet_urls

{'test': ['https://huggingface.co/api/datasets/google/civil_comments/parquet/default/test/0.parquet'],
 'train': ['https://huggingface.co/api/datasets/google/civil_comments/parquet/default/train/0.parquet',
  'https://huggingface.co/api/datasets/google/civil_comments/parquet/default/train/1.parquet'],
 'validation': ['https://huggingface.co/api/datasets/google/civil_comments/parquet/default/validation/0.parquet']}

In [4]:
url_list = []

for split_cat in parquet_urls:
  url_list.extend(parquet_urls[split_cat])

len(url_list)

4

In [6]:
os.makedirs('parquets')

In [5]:
par_id = 0

for url in url_list:
  par_file = requests.get(url).content

  with open('parquets/{}.parquet'.format(par_id), 'wb') as f:
    f.write(par_file)

  par_id += 1

## Data Init

In [6]:
par_files = glob('parquets/*')

df = pd.concat([pd.read_parquet(par_file) for par_file in par_files])

In [7]:
df.head()

,text,toxicity,severe_toxicity,obscene,threat,insult,identity_attack,sexual_explicit
0,Jeff Sessions is another one of Trump's Orwell...,0.200000,0.0,0.000000,0.0,0.200000,0.0,0.0
1,I actually inspected the infrastructure on Gra...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
2,No it won't . That's just wishful thinking on ...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
3,Instead of wringing our hands and nibbling the...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
4,how many of you commenters have garbage piled ...,0.753846,0.0,0.046154,0.0,0.723077,0.0,0.0


In [8]:
df.shape

(1999514, 8)

In [9]:
df = df.iloc[:10000, :]

In [10]:
df_cols = list(df.columns)

for df_col in df_cols:
  if df_col not in ['text', 'toxicity']:
    del df[df_col]

In [11]:
init_samples = df.shape[0]
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
final_samples = df.shape[0]

if final_samples == init_samples:
  print('All sample space reserved')
else:
  print('{} samples were deleted'.format((init_samples - final_samples)))

6 samples were deleted


In [12]:
def preprocess(text):
    text = text.strip()
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()

In [13]:
df['text'] = df['text'].apply(preprocess)

## Data Split

In [14]:
df = df.sample(frac=1)

In [15]:
train_size = df.shape[0] * 8 // 10
train_size

7995

In [16]:
df_train = df.iloc[:train_size, :]
df_test = df.iloc[train_size:, :]

In [17]:
y_train = df_train['toxicity'].values
y_test = df_test['toxicity'].values

# Vectorization

## TF-IDF

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer as tfid

vectorizer = tfid(max_features=100, preprocessor=lambda x: ' '.join(x))

X_train_ti = vectorizer.fit_transform(df_train['text'].values).toarray()
X_test_ti = vectorizer.fit_transform(df_test['text'].values).toarray()

## Pyrrhotite

In [19]:
class Pyrrhotite(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(Pyrrhotite, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.output_layer = nn.Linear(embedding_dim, vocab_size)

    def forward(self, center_words):
        embeds = self.embeddings(center_words)
        out = self.output_layer(embeds)
        return out

In [20]:
base_path = '/content/drive/MyDrive/colabout/pyrrhotite'

In [21]:
with open('{}/vocab.pkl'.format(base_path), 'rb') as f:
    word_to_index = pickle.load(f)
index_to_word = {i: w for w, i in word_to_index.items()}
vocab_size = len(word_to_index)

In [22]:
pyrmodel = Pyrrhotite(vocab_size, 32)
pyrmodel.load_state_dict(torch.load('{}/pyrrhotite.pth'.format(base_path)))
pyrmodel.eval()

Pyrrhotite(
  (embeddings): Embedding(29398, 32)
  (output_layer): Linear(in_features=32, out_features=29398, bias=True)
)

In [23]:
def sentence_to_embedding(sentence, word_to_index, model):
    indices = [word_to_index.get(w, None) for w in sentence]
    indices = [i for i in indices if i is not None]
    if not indices:
        return np.zeros(model.embeddings.embedding_dim)

    input_tensor = torch.tensor(indices, dtype=torch.long)
    with torch.no_grad():
        embeddings = model.embeddings(input_tensor)
        sentence_embedding = embeddings.mean(dim=0)
    return sentence_embedding.numpy()

In [24]:
X_train_pyr = df_train['text'].apply(lambda x: sentence_to_embedding(x, word_to_index, pyrmodel)).values
X_test_pyr = df_test['text'].apply(lambda x: sentence_to_embedding(x, word_to_index, pyrmodel)).values

In [25]:
embedding_dim = pyrmodel.embeddings.embedding_dim

X_train_pyr = np.vstack(X_train_pyr)
X_test_pyr = np.vstack(X_test_pyr)

# Model

In [34]:
!pip install wolta

  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.5
    Uninstalling numpy-1.23.5:
      Successfully uninstalled numpy-1.23.5


In [32]:
!pip install --upgrade numpy==1.23.5

## with TF-IDF

In [26]:
from wolta.model_tools import compare_models

results = compare_models(
    algo_type='reg',
    algorithms=['cat', 'lbm', 'ada', 'raf', 'lin'],
    metrics=['sq', 'abs', 'rlog', 'max'],
    X_train=X_train_ti,
    y_train=y_train,
    X_test=X_test_ti,
    y_test=y_test,
    get_result=True
)

CatBoost
Mean Squared Error: 0.04000972155542882
Mean Absolute Error: 0.14048059694660409
Root Mean Squared Log Error: 0.15715956199013878
Max Error: 0.9272658041671256
***


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


LightGBM
Mean Squared Error: 0.04026043478090862
Mean Absolute Error: 0.1407040938552497
Root Mean Squared Log Error: 0.15824785154952636
Max Error: 0.9235134205091675
***
AdaBoost
Mean Squared Error: 0.05254090668079059
Mean Absolute Error: 0.20248517540346878
Root Mean Squared Log Error: 0.1909426463125406
Max Error: 0.7740308912705659
***
Random Forest
Mean Squared Error: 0.04020124764267834
Mean Absolute Error: 0.1432102987475146
Root Mean Squared Log Error: 0.1581580553517728
Max Error: 0.9373333324491978
***
Linear Regression
Mean Squared Error: 0.03935073989417522
Mean Absolute Error: 0.14235973722893794
Root Mean Squared Log Error: 0.15614608339733593
Max Error: 0.9181149962316308
***


In [27]:
from wolta.model_tools import get_best_model

model = get_best_model(
    results,
    'sq',
    'reg',
    X_train_ti,
    y_train
)

Best Algorithm is lin with the score of 0.03935073989417522


## with Pyrrhotite

In [28]:
from wolta.model_tools import compare_models

results = compare_models(
    algo_type='reg',
    algorithms=['cat', 'lbm', 'ada', 'raf', 'lin'],
    metrics=['sq', 'abs', 'rlog', 'max'],
    X_train=X_train_pyr,
    y_train=y_train,
    X_test=X_test_pyr,
    y_test=y_test,
    get_result=True
)

CatBoost
Mean Squared Error: 0.03889958466816079
Mean Absolute Error: 0.14079474353819652
Root Mean Squared Log Error: 0.1554816044157977
Max Error: 0.9182204591946679
***
LightGBM
Mean Squared Error: 0.039363860539946206
Mean Absolute Error: 0.14148045471618467
Root Mean Squared Log Error: 0.15606553709825796
Max Error: 0.9272405660376175
***


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


AdaBoost
Mean Squared Error: 0.049402271300615516
Mean Absolute Error: 0.1920095915258694
Root Mean Squared Log Error: 0.1841509276204155
Max Error: 0.8291536472438475
***
Random Forest
Mean Squared Error: 0.040750675882664025
Mean Absolute Error: 0.15367782185048262
Root Mean Squared Log Error: 0.16085610390087823
Max Error: 0.9426666655391455
***
Linear Regression
Mean Squared Error: 0.038322373226771615
Mean Absolute Error: 0.1416609772580034
Root Mean Squared Log Error: 0.15319620830125347
Max Error: 0.9012863445771362
***


In [29]:
from wolta.model_tools import get_best_model

model = get_best_model(
    results,
    'sq',
    'reg',
    X_train_pyr,
    y_train
)

Best Algorithm is lin with the score of 0.038322373226771615
